# Generación class-conditional con CFG sobre dígitos a color

Carga el checkpoint condicional `color_digits_VP-Cosine.pth`, genera una rejilla 10×N con muestras para cada dígito (0–9) usando *classifier-free guidance*, y guarda la figura para el informe.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

# Asegúrate de estar en la raíz del proyecto
PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name != 'proyecto_AAIII_02_diffusion_models':
    PROJECT_DIR = PROJECT_DIR.parent
sys.path.insert(0, str(PROJECT_DIR))

from diffusion_lib import UNetScoreModelColor as ScoreNetColor
from diffusion_lib import (
    VPProcess, CosineSchedule,
    EulerMaruyamaSampler,
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## 1. Modelo condicional

Envolvemos `ScoreNetColor` (incondicional) con un *embedding* de clase. El token nulo $\emptyset$ ocupa el índice `n_classes` (= 10).

In [ ]:
class ConditionalScoreNetColor(nn.Module):
    """ScoreNet a color con embedding de clase para CFG."""

    def __init__(self, marginal_prob_std, n_classes=10, embed_dim=256):
        super().__init__()
        self.n_classes  = n_classes
        self.null_token = n_classes
        self.class_embed = nn.Embedding(n_classes + 1, embed_dim)
        self.score_net   = ScoreNetColor(
            marginal_prob_std=marginal_prob_std,
            embed_dim=embed_dim,
        )

    def forward(self, x, t, class_label=None):
        B = x.shape[0]
        if class_label is None:
            class_label = torch.full((B,), self.null_token,
                                     device=x.device, dtype=torch.long)
        c_emb = self.class_embed(class_label)        # [B, embed_dim]
        return self.score_net(x, t, class_emb=c_emb)

## 2. Cargar el checkpoint VP-Cosine condicional

In [ ]:
process     = VPProcess(schedule=CosineSchedule())
cond_model  = ConditionalScoreNetColor(
    marginal_prob_std=process.sigma_t,
    n_classes=10,
).to(device)

ckpt_path = PROJECT_DIR / 'color_digits_cond_checkpoints' / 'color_digits_VP-Cosine.pth'
cond_model.load_state_dict(torch.load(ckpt_path, map_location=device))
cond_model.eval()
print(f'Checkpoint cargado: {ckpt_path.name}')

## 3. Función CFG (combina dos *forward passes*)

Recordatorio:
$$s_\text{CFG}(x,t) = s_\theta(x,t\mid\emptyset) + w\,\bigl[s_\theta(x,t\mid y) - s_\theta(x,t\mid\emptyset)\bigr]$$

In [ ]:
def cfg_score(model, x, t, class_label, cfg_scale):
    """Score CFG: combina pase incondicional y condicional."""
    B = x.shape[0]
    y_cond   = torch.full((B,), int(class_label), device=x.device, dtype=torch.long)
    s_uncond = model(x, t, class_label=None)
    s_cond   = model(x, t, class_label=y_cond)
    return s_uncond + cfg_scale * (s_cond - s_uncond)

## 4. Sampler Euler-Maruyama con guidance

Mismo bucle que el sampler estándar, pero llamando a `cfg_score` en lugar de la red directamente.

In [ ]:
@torch.no_grad()
def cfg_sample(model, process, n_images, class_label, cfg_scale,
               img_shape=(3, 32, 32), n_steps=500, T=0.999, eps=1e-3):
    x = process.prior_sample((n_images, *img_shape), device)
    dt     = (eps - T) / n_steps
    t_vals = torch.linspace(T, eps, n_steps + 1, device=device)
    view_shape = [n_images] + [1] * len(img_shape)

    model.eval()
    for i in range(n_steps):
        t = t_vals[i].expand(n_images)
        score = cfg_score(model, x, t, class_label, cfg_scale)
        drift = process.drift_coefficient(x, t) - process.diffusion_coefficient(t).view(*view_shape)**2 * score
        g_t   = process.diffusion_coefficient(t).view(*view_shape)
        z     = torch.randn_like(x)
        x     = x + drift * dt + g_t * np.sqrt(abs(dt)) * z
    return x

## 5. Rejilla 10 × N: una fila por dígito

Genera $N$ muestras para cada clase y las apila en una rejilla.

In [ ]:
N_PER_CLASS = 6
W           = 3.0

all_samples = []
for y in range(10):
    print(f'Generando clase {y}...')
    sams = cfg_sample(cond_model, process, N_PER_CLASS, class_label=y, cfg_scale=W)
    all_samples.append(sams.cpu())

all_samples = torch.stack(all_samples, dim=0)   # [10, N_PER_CLASS, 3, 32, 32]
print('Forma final:', all_samples.shape)

In [ ]:
fig, axes = plt.subplots(10, N_PER_CLASS, figsize=(N_PER_CLASS * 1.4, 14))
for y in range(10):
    for k in range(N_PER_CLASS):
        ax = axes[y, k]
        img = all_samples[y, k].permute(1, 2, 0).clamp(0, 1).numpy()
        ax.imshow(img)
        ax.axis('off')
        if k == 0:
            ax.set_ylabel(f'y={y}', rotation=0, labelpad=20, fontsize=11)

fig.suptitle(f'Class-conditional con CFG (w={W}) — VP-Cosine sobre color digits', y=0.92)
plt.tight_layout()

out_dir = PROJECT_DIR / 'figuras'
out_dir.mkdir(exist_ok=True)
fig.savefig(out_dir / 'cfg_grid_color_digits.pdf', bbox_inches='tight', dpi=200)
print(f'Figura guardada en {out_dir / "cfg_grid_color_digits.pdf"}')
plt.show()

## 6. Sanity check

Verifica que el modelo discrimina entre clases. Si la diferencia $\|s_y - s_\emptyset\|$ varía con $y$, la red ha aprendido las clases.

In [ ]:
with torch.no_grad():
    x_dummy = torch.randn(1, 3, 32, 32, device=device)
    t_dummy = torch.tensor([0.5], device=device)
    s_uncond = cond_model(x_dummy, t_dummy, class_label=None)
    norms = []
    for y in range(10):
        y_t = torch.tensor([y], device=device, dtype=torch.long)
        s_y = cond_model(x_dummy, t_dummy, class_label=y_t)
        norms.append((s_y - s_uncond).norm().item())

print('|| s_y − s_∅ || por clase:')
for y, n in enumerate(norms):
    print(f'  y={y}: {n:.4f}')